In [ ]:
from Methods.CodetT5.gptsniffer import CodeT5pClassifier
from Methods.CodetT5.dataset import CodeDataset

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

In [4]:
# Define the training dataset and dataloader
import warnings
import torch
from torch.utils.data import DataLoader, Dataset
from transformers import T5EncoderModel, Trainer, TrainingArguments

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
#device = torch.device('cpu')
if device == 'cpu':
    warnings.warn("Using cpu because cuda not available")




train_dataset = CodeDataset('./Dataset/CodeMirage_train.csv')
train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True)

# Define the testing dataset and dataloader
test_dataset = CodeDataset('./Dataset/CodeMirage_test.csv')
test_dataloader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# Define the training arguments and the trainer
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=12,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=10,
    optim='adamw_torch',
    learning_rate=5e-5,
    save_total_limit=2,
    # metric_for_best_model='f1',
    # report_to='wandb',
    # push_to_hub=True,
    # hub_strategy='every_save',
    # hub_model_id=repository_id,
    # hub_token=HfFolder.get_token(),
)


model = CodeT5pClassifier().to(device)


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset
    # eval_dataset=test_dataset,
)

# Train the model with the pre-defined parameters
trainer.train()

config.json:   0%|          | 0.00/768 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/446M [00:00<?, ?B/s]

  0%|          | 0/2700 [00:00<?, ?it/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 1.50 GiB. GPU 0 has a total capacity of 11.99 GiB of which 0 bytes is free. Of the allocated memory 25.11 GiB is allocated by PyTorch, and 112.00 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
import numpy as np
import torch.nn.functional as F
from sklearn.metrics import roc_curve, roc_auc_score, RocCurveDisplay

model.eval()
y_true, y_score = [], []

with torch.no_grad():
    for batch in test_dataloader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        out = model(input_ids, attention_mask=attention_mask)
        logits = out["logits"] if isinstance(out, dict) else out.logits
        probs = F.softmax(logits, dim=1)          # shape [B,2]
        y_true.extend(labels.cpu().numpy().tolist())
        y_score.extend(probs[:, 1].cpu().numpy().tolist())  # 1 = classe positiva
# ROC + AUC
y_true = np.array(y_true); y_score = np.array(y_score)
fpr, tpr, thr = roc_curve(y_true, y_score, pos_label=1)
auc_roc = roc_auc_score(y_true, y_score)
print(f"AUC-ROC: {auc_roc:.4f}")

# plot veloce (opzionale)
RocCurveDisplay.from_predictions(y_true, y_score)


In [ ]:
from sklearn.metrics import confusion_matrix
import time
start = time.time()
#cm = confusion_matrix(y_true, y_pred)
end = time.time()
print(f'time to inference: {end - start} seconds')

#print(cm)




























##########################################################################################################
#Plot the confusion matrix. Set Normalize = True/False

def plot_confusion_matrix(cm, classes, normalize=True, title='Confusion matrix', cmap=plt.cm.Blues):
    """
    This function prints and plots the confusion matrix.
    Normalization can be applied by setting `normalize=True`.
    """
    plt.figure(figsize=(20,20))
    plt.imshow(cm, interpolation='nearest', cmap=cmap)
    plt.title(title)
    plt.colorbar()
    tick_marks = np.arange(len(classes))
    plt.xticks(tick_marks, classes, rotation=45)
    plt.yticks(tick_marks, classes)
    if normalize:
        cm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
        cm = np.around(cm, decimals=2)
        cm[np.isnan(cm)] = 0.0
        print("Normalized confusion matrix")
    else:
        print('Confusion matrix, without normalization')
    thresh = cm.max() / 2.
    for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
        plt.text(j, i, cm[i, j],
                 horizontalalignment="center",
                 color="white" if cm[i, j] > thresh else "black")
    plt.tight_layout()
    plt.ylabel('True label')
    plt.xlabel('Predicted label')
    

#Print the Target names
from sklearn.metrics import classification_report, confusion_matrix
import itertools 
#shuffle=False
target_names = ['ChatGPT','Human']

print('Confusion Matrix')
cm = confusion_matrix(y_true, y_pred)
plot_confusion_matrix(cm, target_names, title='Confusion Matrix')
# Print Classification Report
print('Classification Report')
print(classification_report(y_true, y_pred, target_names=target_names))